**Importe**

In [1]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torch.nn as nn
from torchvision import transforms 

from sklearn.metrics import f1_score

from training_monitor import TrainingMonitor
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

<jemalloc>: Unsupported system page size


- Seed setzen um spätere Vergleichbarkeit, der Zufallsproben zu schaffen
- Cache der Grafikkarte leeren, um OOM zu vermeiden
- Cuda:2 in der Variable device deklarieren, sodass ich, falls ich auf einer anderen GPU arbeiten will, diese nur einmal hier ändern muss

In [2]:
import torch, gc, random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=2)

Pfade für Bilder und CSVs deklarieren und in Dataframes lesen.

In [ ]:
train1_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_path   = "/datasets/multi-view-pig-posture-recognition/test_images"

train1_csv  = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv  = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv    = "/datasets/multi-view-pig-posture-recognition/test.csv"

import pandas as pd

train1_df = pd.read_csv(train1_csv)
train2_df = pd.read_csv(train2_csv)
test_df   = pd.read_csv(test_csv)

train1_df.head()

In [ ]:
test_df.head()


Klassen aus .txt korrekt einlesen und ebenfalls in Dataframe speichern. Zusätzlich werden die Spaten benannt und class_id vergeben.

In [ ]:
pig_posture_classes = pd.read_csv(
    "/datasets/multi-view-pig-posture-recognition/pig_posture_classes.txt",
    header=None,
    names=["class_name"]
)
pig_posture_classes["class_id"] = pig_posture_classes.index
pig_posture_classes.head()

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import ast
import os
from functools import lru_cache
from PIL import Image
from torch.utils.data import Dataset
import torch

# ── Augmentations ──
train_aug = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['class_id']))

val_aug = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['class_id']))

# ── Dataset ──
class PigDataset(Dataset):
    def __init__(self, df, image_root, transform=None, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform
        self.has_labels = has_labels

    @staticmethod
    @lru_cache(maxsize=256)
    def _load_image(path):
        return Image.open(path).convert("RGB")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, row["image_id"])
        img_np = np.array(self._load_image(img_path))

        if self.has_labels:
            bbox = ast.literal_eval(row["bbox"])       # [x_min, y_min, w, h]
            label = int(row["class_id"])

            if self.transform:
                transformed = self.transform(
                    image=img_np,
                    bboxes=[bbox],
                    class_id=[label]
                )
                img = transformed["image"]
                bbox = torch.tensor(transformed["bboxes"][0], dtype=torch.float32)
                label = torch.tensor(transformed["class_id"][0], dtype=torch.long)
            else:
                img = torch.from_numpy(img_np).permute(2, 0, 1).float()
                bbox = torch.tensor(bbox, dtype=torch.float32)
                label = torch.tensor(label, dtype=torch.long)

            return img, label, bbox
        else:
            if self.transform:
                transformed = self.transform(image=img_np, bboxes=[], class_id=[])
                img = transformed["image"]
            else:
                img = torch.from_numpy(img_np).permute(2, 0, 1).float()
            return img

In [ ]:
from sklearn.model_selection import train_test_split

unique_images = train1_df["image_id"].unique()
train_imgs, val_imgs = train_test_split(unique_images, test_size=0.2, random_state=42)

train_df = train1_df[train1_df["image_id"].isin(train_imgs)].reset_index(drop=True)
val_df   = train1_df[train1_df["image_id"].isin(val_imgs)].reset_index(drop=True)

train_dataset = PigDataset(train_df, train1_path, transform=train_aug, has_labels=True)
val_dataset   = PigDataset(val_df,   train1_path, transform=val_aug,   has_labels=True)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=8, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=8, pin_memory=False)

In [ ]:
class TwoHeadModel(nn.Module):
    """
    2-Head-Modell:
      - Backbone: vortrainiertes ResNet50 (Feature-Extraktor)
      - Head 1:   Classification  → num_classes Klassen
      - Head 2:   BBox Regression → 4 Werte (x, y, w, h) normalisiert [0,1]
    """

    def __init__(self, num_classes: int, backbone_name: str = "resnet50"):
        super().__init__()

        # ── Backbone (vortrainiert) ──
        weights = models.ResNet50_Weights.DEFAULT
        backbone_full = models.resnet50(weights=weights)

        # Alles außer den letzten FC-Layer als Feature-Extraktor
        self.backbone = nn.Sequential(*list(backbone_full.children())[:-1])
        # Output: (batch, 2048, 1, 1)

        feature_dim = 2048  # ResNet50 → 2048 Features

        # ── Head 1: Classification ──
        self.cls_head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(feature_dim, num_classes),
        )

        # ── Head 2: BBox Regression ──
        self.bbox_head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid(),  # Werte zwischen 0 und 1
        )

    def forward(self, x):
        features = self.backbone(x)          # (B, 2048, 1, 1)
        cls_logits = self.cls_head(features)
        bbox_pred = self.bbox_head(features)
        return cls_logits, bbox_pred

In [ ]:
class CombinedLoss(nn.Module):
    """
    Kombinierter Loss:
      - CrossEntropy für Classification
      - SmoothL1 (Huber) für BBox Regression
      - alpha gewichtet den BBox-Anteil
    """

    def __init__(self, alpha: float = 1.0):
        super().__init__()
        self.cls_loss_fn = nn.CrossEntropyLoss()
        self.bbox_loss_fn = nn.SmoothL1Loss()
        self.alpha = alpha

    def forward(self, cls_logits, bbox_pred, cls_target, bbox_target):
        cls_loss = self.cls_loss_fn(cls_logits, cls_target)
        bbox_loss = self.bbox_loss_fn(bbox_pred, bbox_target)
        total = cls_loss + self.alpha * bbox_loss
        return total, cls_loss, bbox_loss

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from training_monitor import TrainingMonitor


# ──────────────────────────────────────────────
# Hilfsfunktion: IoU berechnen
# ──────────────────────────────────────────────
def compute_iou(pred_bbox, target_bbox):
    """
    Berechnet IoU für Bounding Boxes im Format [x, y, w, h] (normalisiert 0-1).
    Ändert die Eingabe-Tensoren NICHT – gibt einen neuen Tensor zurück.

    Args:
        pred_bbox:   (N, 4) Tensor – vorhergesagte Boxen
        target_bbox: (N, 4) Tensor – wahre Boxen

    Returns:
        (N,) Tensor – IoU pro Sample
    """
    # [x, y, w, h] → [x1, y1, x2, y2]
    pred_x1 = pred_bbox[:, 0]
    pred_y1 = pred_bbox[:, 1]
    pred_x2 = pred_bbox[:, 0] + pred_bbox[:, 2]
    pred_y2 = pred_bbox[:, 1] + pred_bbox[:, 3]

    tgt_x1 = target_bbox[:, 0]
    tgt_y1 = target_bbox[:, 1]
    tgt_x2 = target_bbox[:, 0] + target_bbox[:, 2]
    tgt_y2 = target_bbox[:, 1] + target_bbox[:, 3]

    inter_x1 = torch.max(pred_x1, tgt_x1)
    inter_y1 = torch.max(pred_y1, tgt_y1)
    inter_x2 = torch.min(pred_x2, tgt_x2)
    inter_y2 = torch.min(pred_y2, tgt_y2)

    inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    pred_area = (pred_x2 - pred_x1) * (pred_y2 - pred_y1)
    tgt_area = (tgt_x2 - tgt_x1) * (tgt_y2 - tgt_y1)

    union = pred_area + tgt_area - inter_area
    return inter_area / (union + 1e-6)


class Learner:
    def __init__(
        self,
        model,
        train_dl,
        val_dl,
        cls_loss_fn=None,
        bbox_loss_fn=None,
        bbox_loss_weight=1.0,
        eval_steps=50,
        device=device,
        amp=False,
    ):
        """
        Multi-Task Learner für Klassifikation + BBox-Regression.

        Das Modell muss ein Dict {"cls": Tensor, "bbox": Tensor}
        oder ein Tuple (cls_logits, bbox_pred) zurückgeben.

        Die DataLoader müssen (img, label, bbox) liefern.

        Args:
            model:            nn.Module mit Multi-Task-Output.
            train_dl:         DataLoader → (img, label, bbox).
            val_dl:           DataLoader → (img, label, bbox).
            cls_loss_fn:      Loss für Klassifikation (default: CrossEntropyLoss).
            bbox_loss_fn:     Loss für BBox-Regression (default: SmoothL1Loss).
            bbox_loss_weight: Gewichtung des BBox-Loss relativ zum Cls-Loss.
            eval_steps:       Alle N Steps evaluieren + Monitor updaten.
            device:           "cuda", "cpu" oder None (auto).
            amp:              Mixed Precision aktivieren.
        """
        self.model = model
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.cls_loss_fn = cls_loss_fn or nn.CrossEntropyLoss()
        self.bbox_loss_fn = bbox_loss_fn or nn.SmoothL1Loss()
        self.bbox_loss_weight = bbox_loss_weight

        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)

        self.eval_steps = eval_steps
        self.amp = bool(amp) and self.device.type == "cuda"
        self.scaler = torch.cuda.amp.GradScaler(enabled=self.amp)
        self.model.to(self.device)

        # ── Tracking ──
        self.best_val_f1 = -1.0
        self.best_state = None

    # ─────────────────────────────────────────────
    # Model Output parsen
    # ─────────────────────────────────────────────
    @staticmethod
    def _parse_output(output):
        """Akzeptiert Dict {"cls": ..., "bbox": ...} oder Tuple (cls, bbox)."""
        if isinstance(output, dict):
            return output["cls"], output["bbox"]
        elif isinstance(output, (tuple, list)):
            return output[0], output[1]
        else:
            raise ValueError(
                f"Modell muss dict oder tuple zurückgeben, nicht {type(output)}. "
                f"Erwarte {{'cls': Tensor, 'bbox': Tensor}} oder (cls, bbox)."
            )

    # ─────────────────────────────────────────────
    # Combined Loss
    # ─────────────────────────────────────────────
    def _compute_loss(self, cls_logits, bbox_pred, labels, bboxes):
        cls_loss = self.cls_loss_fn(cls_logits, labels)
        bbox_loss = self.bbox_loss_fn(bbox_pred, bboxes)
        total_loss = cls_loss + self.bbox_loss_weight * bbox_loss
        return total_loss, cls_loss, bbox_loss

    # ─────────────────────────────────────────────
    # Freeze / Unfreeze
    # ─────────────────────────────────────────────
    def freeze(self):
        if hasattr(self.model, "backbone"):
            for p in self.model.backbone.parameters():
                p.requires_grad = False
            print("✅ Backbone eingefroren.")
        else:
            print("⚠️  Kein 'backbone'-Attribut gefunden.")

    def unfreeze(self):
        for p in self.model.parameters():
            p.requires_grad = True
        print("✅ Alle Parameter freigegeben.")

    # ─────────────────────────────────────────────
    # Evaluation
    # ─────────────────────────────────────────────
    @torch.no_grad()
    def evaluate(self):
        """
        Evaluiert auf dem Validation-Set.

        Returns:
            dict mit val_loss, val_cls_loss, val_bbox_loss, val_macro_f1, val_iou
        """
        self.model.eval()
        total_loss = 0.0
        total_cls_loss = 0.0
        total_bbox_loss = 0.0
        total_samples = 0
        all_preds = []
        all_labels = []
        all_ious = []

        for xb, yb_cls, yb_bbox in self.val_dl:
            xb = xb.to(self.device, non_blocking=True)
            yb_cls = yb_cls.to(self.device, non_blocking=True)
            yb_bbox = yb_bbox.to(self.device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=self.amp):
                output = self.model(xb)
                cls_logits, bbox_pred = self._parse_output(output)
                loss, cls_loss, bbox_loss = self._compute_loss(
                    cls_logits, bbox_pred, yb_cls, yb_bbox
                )

            bs = xb.size(0)
            total_loss += loss.item() * bs
            total_cls_loss += cls_loss.item() * bs
            total_bbox_loss += bbox_loss.item() * bs
            total_samples += bs

            all_preds.append(cls_logits.argmax(dim=1).cpu())
            all_labels.append(yb_cls.cpu())
            all_ious.append(compute_iou(bbox_pred.cpu(), yb_bbox.cpu()))

        val_loss = total_loss / total_samples
        val_cls_loss = total_cls_loss / total_samples
        val_bbox_loss = total_bbox_loss / total_samples

        all_preds = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        val_f1 = f1_score(all_labels, all_preds, average="macro")
        val_iou = torch.cat(all_ious).mean().item()

        self.model.train()
        return {
            "val_loss": val_loss,
            "val_cls_loss": val_cls_loss,
            "val_bbox_loss": val_bbox_loss,
            "val_macro_f1": val_f1,
            "val_iou": val_iou,
        }

    # ─────────────────────────────────────────────
    # Interval-Tracking Helfer
    # ─────────────────────────────────────────────
    @staticmethod
    def _new_interval():
        return {
            "loss": 0.0,
            "cls_loss": 0.0,
            "bbox_loss": 0.0,
            "iou_sum": 0.0,
            "count": 0,
            "preds": [],
            "labels": [],
        }

    # ─────────────────────────────────────────────
    # Internes Eval + Monitor + Best-Tracking
    # ─────────────────────────────────────────────
    def _eval_and_track(self, interval, monitor):
        n = max(interval["count"], 1)
        train_loss = interval["loss"] / n
        train_cls_loss = interval["cls_loss"] / n
        train_bbox_loss = interval["bbox_loss"] / n
        train_iou = interval["iou_sum"] / n

        if interval["preds"] and interval["labels"]:
            t_preds = torch.cat(interval["preds"]).numpy()
            t_labels = torch.cat(interval["labels"]).numpy()
            train_f1 = f1_score(t_labels, t_preds, average="macro")
        else:
            train_f1 = 0.0

        val_results = self.evaluate()

        monitor.update({
            "train_loss":      train_loss,
            "train_cls_loss":  train_cls_loss,
            "train_bbox_loss": train_bbox_loss,
            "train_macro_f1":  train_f1,
            "train_iou":       train_iou,
            **val_results,
        })

        if val_results["val_macro_f1"] > self.best_val_f1:
            self.best_val_f1 = val_results["val_macro_f1"]
            self.best_state = {
                k: v.cpu().clone()
                for k, v in self.model.state_dict().items()
            }

        return val_results

    # ─────────────────────────────────────────────
    # Training
    # ─────────────────────────────────────────────
    def fit(
        self,
        epochs,
        lr,
        weight_decay=1e-3,
        use_onecycle=True,
        pct_start=0.3,
        div_factor=25.0,
        final_div_factor=1e4,
        max_grad_norm=1.0,
    ):
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)

        total_steps = epochs * len(self.train_dl)
        steps_per_epoch = len(self.train_dl)

        if use_onecycle:
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=lr, total_steps=total_steps,
                pct_start=pct_start, div_factor=div_factor,
                final_div_factor=final_div_factor,
            )
        else:
            scheduler = None

        # ── TrainingMonitor ──
        num_interval_evals = total_steps // self.eval_steps
        epoch_end_extras = sum(
            1 for e in range(1, epochs + 1)
            if (e * steps_per_epoch) % self.eval_steps != 0
        )
        num_evals = num_interval_evals + epoch_end_extras

        metric_groups = {
            "Total Loss":  ["train_loss", "val_loss"],
            "Cls Loss":    ["train_cls_loss", "val_cls_loss"],
            "BBox Loss":   ["train_bbox_loss", "val_bbox_loss"],
            "Macro F1":    ["train_macro_f1", "val_macro_f1"],
            "IoU":         ["train_iou", "val_iou"],
        }
        monitor = TrainingMonitor(
            total_iterations=max(1, num_evals),
            plot_mode="separate",
            metric_groups=metric_groups,
            zoom=True,
            zoom_factor=0.5,
        )

        print(f"{'='*60}")
        print(f"Training: {epochs} Epochs, {steps_per_epoch} Batches/Epoch, "
              f"{total_steps} Total Steps")
        print(f"LR={lr}, WD={weight_decay}, AMP={self.amp}, Device={self.device}")
        print(f"BBox Loss Weight={self.bbox_loss_weight}")
        print(f"Eval alle {self.eval_steps} Steps → ~{num_evals} Monitor-Updates")
        print(f"{'='*60}")

        self.model.train()
        step = 0

        for epoch in range(1, epochs + 1):
            interval = self._new_interval()
            did_eval_at_epoch_end = False

            for xb, yb_cls, yb_bbox in self.train_dl:
                xb = xb.to(self.device, non_blocking=True)
                yb_cls = yb_cls.to(self.device, non_blocking=True)
                yb_bbox = yb_bbox.to(self.device, non_blocking=True)

                with torch.cuda.amp.autocast(enabled=self.amp):
                    output = self.model(xb)
                    cls_logits, bbox_pred = self._parse_output(output)
                    loss, cls_loss, bbox_loss = self._compute_loss(
                        cls_logits, bbox_pred, yb_cls, yb_bbox
                    )

                optimizer.zero_grad()
                self.scaler.scale(loss).backward()

                if max_grad_norm is not None:
                    self.scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable, max_grad_norm)

                self.scaler.step(optimizer)
                self.scaler.update()

                if scheduler is not None:
                    scheduler.step()

                bs = xb.size(0)
                interval["loss"] += loss.item() * bs
                interval["cls_loss"] += cls_loss.item() * bs
                interval["bbox_loss"] += bbox_loss.item() * bs
                interval["count"] += bs

                with torch.no_grad():
                    batch_iou = compute_iou(bbox_pred.cpu(), yb_bbox.cpu()).sum().item()
                interval["iou_sum"] += batch_iou
                interval["preds"].append(cls_logits.argmax(dim=1).detach().cpu())
                interval["labels"].append(yb_cls.detach().cpu())

                step += 1

                # ── Periodische Evaluation ──
                if step % self.eval_steps == 0:
                    self._eval_and_track(interval, monitor)
                    interval = self._new_interval()

                    if step == epoch * steps_per_epoch:
                        did_eval_at_epoch_end = True

            # ── Epoch-Ende: nur evaluieren wenn nicht gerade passiert ──
            if not did_eval_at_epoch_end:
                self._eval_and_track(interval, monitor)
                interval = self._new_interval()

            # Epoch-Zusammenfassung
            val_results = self.evaluate()
            print(
                f"  Epoch {epoch}/{epochs} | "
                f"Val Loss: {val_results['val_loss']:.4f} | "
                f"Val F1: {val_results['val_macro_f1']:.4f} | "
                f"Val IoU: {val_results['val_iou']:.4f}"
            )

        print(f"{'='*60}")
        print(f"Training fertig. Bester Val Macro-F1: {self.best_val_f1:.4f}")
        print(f"{'='*60}")
        return self.best_state, self.best_val_f1

    # ─────────────────────────────────────────────
    # Best Model laden
    # ─────────────────────────────────────────────
    def load_best(self):
        if self.best_state is not None:
            self.model.load_state_dict(self.best_state)
            self.model.to(self.device)
            print(f"✅ Bestes Modell geladen (F1: {self.best_val_f1:.4f})")
        else:
            print("⚠️  Kein bestes Modell gespeichert.")

    # ─────────────────────────────────────────────
    # Predictions
    # ─────────────────────────────────────────────
    @torch.no_grad()
    def predict(self, dl):
        """
        Gibt (class_preds, bbox_preds) als numpy arrays zurück.
        """
        self.model.eval()
        all_cls = []
        all_bbox = []

        for batch in dl:
            xb = batch[0].to(self.device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=self.amp):
                output = self.model(xb)
                cls_logits, bbox_pred = self._parse_output(output)

            all_cls.append(cls_logits.argmax(dim=1).cpu())
            all_bbox.append(bbox_pred.cpu())

        self.model.train()
        return torch.cat(all_cls).numpy(), torch.cat(all_bbox).numpy()

In [ ]:
num_classes = train1_df["class_id"].nunique()
print(num_classes)

In [ ]:
# ── Modell muss (cls_logits, bbox_pred) oder {"cls": ..., "bbox": ...} zurückgeben ──
model = TwoHeadModel(num_classes=5).to(device)

learner = Learner(
    model=model,
    train_dl=train_loader,
    val_dl=val_loader,
    bbox_loss_weight=1.0,   # Gewichtung BBox-Loss vs Cls-Loss
    eval_steps=20,          # alle 20 Batches evaluieren + Monitor updaten
)

# Phase 1: Backbone einfrieren, nur Heads trainieren
learner.freeze()
best_state, best_f1 = learner.fit(epochs=5, lr=1e-3)



In [ ]:
def show_predictions(model, dl, class_names, device="cuda:2", n=8):
    """
    Zeigt n Bilder mit predicted BBox (rot) und true BBox (grün).
    """
    model.eval()
    model.to(device)

    imgs, labels, bboxes = next(iter(dl))
    imgs_dev = imgs.to(device)

    with torch.no_grad():
        cls_logits, bbox_pred = model(imgs_dev)

    pred_labels = cls_logits.argmax(dim=1).cpu().numpy()
    pred_bboxes = bbox_pred.cpu().numpy()
    true_labels = labels.numpy()
    true_bboxes = bboxes.numpy()

    n = min(n, len(imgs))
    cols = max(n // 2, 1)
    fig, axes = plt.subplots(2, cols, figsize=(4 * cols, 8))
    axes = axes.flatten()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    for i in range(n):
        img_np = imgs[i].permute(1, 2, 0).numpy()
        img_np = std * img_np + mean
        img_np = np.clip(img_np, 0, 1)

        h, w = img_np.shape[:2]
        ax = axes[i]
        ax.imshow(img_np)

        # Predicted BBox (rot)
        px, py, pw, ph = pred_bboxes[i]
        rect_pred = patches.Rectangle(
            (px * w, py * h), pw * w, ph * h,
            linewidth=2, edgecolor="red", facecolor="none", linestyle="--"
        )
        ax.add_patch(rect_pred)

        # True BBox (grün)
        tx, ty, tw, th = true_bboxes[i]
        rect_true = patches.Rectangle(
            (tx * w, ty * h), tw * w, th * h,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect_true)

        pred_name = class_names[pred_labels[i]]
        true_name = class_names[true_labels[i]]
        color = "green" if pred_labels[i] == true_labels[i] else "red"
        ax.set_title(f"Pred: {pred_name}\nTrue: {true_name}", color=color, fontsize=10)
        ax.axis("off")

    legend_elements = [
        patches.Patch(edgecolor="lime", facecolor="none", label="True BBox"),
        patches.Patch(edgecolor="red", facecolor="none", linestyle="--", label="Pred BBox"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Phase 2: Alles freigeben, Fine-Tuning
learner.unfreeze()
best_state, best_f1 = learner.fit(epochs=10, lr=1e-4)


In [ ]:
# Bestes Modell laden
learner.load_best()

In [ ]:
# Predictions
cls_preds, bbox_preds = learner.predict(test_dl)